# twbt：台股回測工具組（Colab 版）

程式和資料都直接從 GitHub（[kurochu888/twbt](https://github.com/kurochu888/twbt)）取得，資料每個交易日由 GitHub 自動更新。

不想寫程式的話，用網頁版就好：https://kurochu888.github.io/twbt/

從上到下按 ▶ 執行。

## 1. 取得最新的程式與資料

In [ ]:
import os, subprocess, sys
REPO = '/content/twbt'
if os.path.exists(f'{REPO}/.git'):
    print(subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], capture_output=True, text=True).stdout)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kurochu888/twbt', REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
DATA = f'{REPO}/data'
print(subprocess.run(['git', 'log', '-1', '--format=最新版本：%cd  %s', '--date=format:%Y-%m-%d %H:%M'], capture_output=True, text=True).stdout)

## 2. 跑測試（不需連網）
最後看到 `OK` 就代表程式正常。

In [ ]:
!python -m unittest discover tests 2>&1 | tail -3

## 3. 計算還原價與模擬序列

從 GitHub 上的原始資料重建，幾秒鐘就好，不用連證交所。

`DOWNLOAD = True` 會另外向證交所抓最新資料（通常不需要，GitHub 每天會自動更新）。

In [ ]:
#@title
DOWNLOAD = False #@param {type:"boolean"}
flag = '' if DOWNLOAD else '--build-only'
!python scripts/update_core.py {flag} --web '' 2>&1 | tail -12

## 4. 看看資料

In [ ]:
import pandas as pd
A = f'{DATA}/adjusted'
print(open(f'{A}/long/calibration.txt', encoding='utf-8').read())
for f in ['0050', '00631L', '00675L']:
    x = pd.read_csv(f'{A}/long/{f}_long.csv', index_col=0, parse_dates=True)
    listed = x.index[x.index.get_loc(x[x.is_synthetic == 0].index[0]) - 1]   # 上市日
    print(f'{f}：{x.index[0].date()} ~ {x.index[-1].date()}，上市日 {listed.date()}')
m = pd.read_csv(f'{A}/long/TAIEX_long.csv', index_col=0, parse_dates=True)
ax = m['tr_index'].plot(logy=True, figsize=(10, 4), title='TAIEX total return index (log scale, pre-2003 estimated)')

# 回測：買進持有

**共同設定**（每次改完都要重新執行這格）：
- `CAPITAL`：投入金額
- `DISCOUNT`：手續費折扣，1 = 不打折，0.28 = 2.8 折
- `USE_SYNTHETIC`：是否使用上市前的模擬資料（關掉 = 只用真實資料）
- `TW50_EXCESS`：模擬段的 0050、00631L 是否套用「台灣50 贏大盤」的校正差距。關掉 = 保守版，假設上市前台灣50 和大盤表現相同

In [ ]:
#@title 5. 回測共同設定
CAPITAL = 1000000 #@param {type:"integer"}
DISCOUNT = 1.0 #@param {type:"number"}
USE_SYNTHETIC = True #@param {type:"boolean"}
TW50_EXCESS = True #@param {type:"boolean"}

import warnings; warnings.simplefilter('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from twbt.backtest import BuyHold, TaiwanCost, describe, worst_best, win_rate
from twbt.backtest.report import fmt_stats, fmt_rows
from twbt.backtest.plots import plot_equity, plot_windows

bh = BuyHold(f'{DATA}/adjusted', use_synthetic=USE_SYNTHETIC, capital=CAPITAL,
             cost=TaiwanCost(discount=DISCOUNT), tw50_excess=TW50_EXCESS)
print(f'資料期間：{bh.rets.index[0].date()} ~ {bh.rets.index[-1].date()}')

In [ ]:
#@title 6. 自訂區間
START = '2000-01-01' #@param {type:"date"}
END = '2010-12-31' #@param {type:"date"}

stats, curves = bh.period(START, END)
show = ['起始日', '結束日', '年數', '總報酬', '年化報酬', '年化波動', '最大回撤', '回撤高點', '回撤低點',
        '回到前高', '夏普', '最好年度', '最差年度', '期末金額', '模擬資料占比']
display(fmt_stats(stats.loc[show]))
plot_equity(curves, f'Buy & Hold {START} ~ {END}'); plt.show()

In [ ]:
#@title 7. 固定長度滾動（每個月一個起點）
YEARS = 10 #@param {type:"number"}

roll = bh.rolling(YEARS)
print(f'{YEARS} 年滾動，每個月初一個起點')
print('\n【年化報酬分布】'); display(fmt_stats(describe(roll, '年化報酬')))
print('\n【最大回撤分布】'); display(fmt_stats(describe(roll, '最大回撤')))
print('\n【最差與最好的區間】'); display(fmt_rows(worst_best(roll)))
for a in ['00631L 台灣50正2', '00675L 加權正2', '大盤（含息）']:
    print(f'{a} 年化報酬勝過 0050 的比例：{win_rate(roll, a, "0050"):.1%}')
plot_windows(roll, title=f'{YEARS:g}-year rolling CAGR'); plt.show()
plot_windows(roll, '最大回撤', title=f'{YEARS:g}-year rolling max drawdown'); plt.show()

In [ ]:
#@title 8. 隨機區間
N = 500 #@param {type:"integer"}
YEARS_MIN = 5 #@param {type:"number"}
YEARS_MAX = 15 #@param {type:"number"}
SEED = 42 #@param {type:"integer"}
#@markdown SEED 固定時每次結果相同；改成 -1 則每次隨機不同。

rnd = bh.random(N, (YEARS_MIN, YEARS_MAX), seed=None if SEED < 0 else SEED)
print(f'隨機 {N} 個區間，長度 {YEARS_MIN}～{YEARS_MAX} 年')
print('\n【年化報酬分布】'); display(fmt_stats(describe(rnd, '年化報酬')))
print('\n【最大回撤分布】'); display(fmt_stats(describe(rnd, '最大回撤')))
print('\n【最差與最好的區間】'); display(fmt_rows(worst_best(rnd)))
plot_windows(rnd, title=f'Random windows ({YEARS_MIN:g}-{YEARS_MAX:g} years) CAGR'); plt.show()